Meet-the-Neighbors Colab notebook.

This file is written in the same style as a Colab-exported Python notebook.
Open it in Google Colab, then use Runtime -> Run all after adjusting the
form fields.

# Meet-the-Neighbors: genome neighborhood VF prediction

This notebook runs the genome-only `meetneighbors predictvf` workflow on a
small upload set. It is intended for Google Colab, where RAM, disk, and GPU
time are limited.

## Inputs

Upload one pair of files per genome:

* One `.gff` annotation file.
* One matching protein FASTA file ending in `.faa` or `.fasta`.

The files are paired by shared basename. These are valid examples:

* `GenomeA.gff` and `GenomeA.faa`
* `GenomeB.gff` and `GenomeB.fasta`

By default, this notebook accepts up to 3 genomes. Change `MAX_GENOMES` in the
setup cell if you have a larger Colab runtime and want to allow more.

## Output

Only `neighborhood_based_predictions.tsv` is kept and downloaded. Uploaded
files, the cloned repository, and intermediate pipeline files are removed at
the end to save Colab disk space.

## Install dependencies

This cell clones Meet-the-Neighbors, installs the Python package, and installs
the command-line tools required by the pipeline.

The install can take several minutes. If the runtime disconnects or restarts,
rerun the notebook from this cell.

In [1]:
#@title 1. Install Meet-the-Neighbors, MMseqs2, Foldseek, and gLM weights

from pathlib import Path
import importlib.util
import os
import platform
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/mcn3159/meet-the-neighbors.git"
REPO_REF = "colab_run"

WORK_ROOT = Path("/content/meetneighbors_colab")
TOOLS_DIR = WORK_ROOT / "tools"
UPLOAD_DIR = WORK_ROOT / "uploaded_genomes"
RUN_DIR = WORK_ROOT / "run"
OUTPUT_DIR = RUN_DIR
FINAL_DIR = WORK_ROOT / "final_results"
GENOME_TSV = RUN_DIR / "genome_pairs.tsv"
FINAL_OUTPUT = FINAL_DIR / "neighborhood_based_predictions.tsv"

MAX_GENOMES = 3
threads = 2
mem_gb = 12
glm_batch_size = 50

WORK_ROOT.mkdir(parents=True, exist_ok=True)
TOOLS_DIR.mkdir(parents=True, exist_ok=True)

# Make tools visible to Python subprocesses in this notebook session.
os.environ["PATH"] = f"{TOOLS_DIR}:{os.environ.get('PATH', '')}"


def run_cmd(cmd, cwd=None):
    printable = " ".join(str(x) for x in cmd)
    print(f"$ cd {cwd} && {printable}" if cwd else f"$ {printable}")
    subprocess.run([str(x) for x in cmd], cwd=cwd, check=True)


def has_cpu_flag(flag: str) -> bool:
    try:
        return flag in Path("/proc/cpuinfo").read_text()
    except Exception:
        return False


def download(url: str, dest: Path):
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size > 0:
        print(f"Already downloaded: {dest}")
        return

    if shutil.which("aria2c"):
        run_cmd([
            "aria2c",
            "-x", "16",
            "-s", "16",
            "-k", "1M",
            "--file-allocation=none",
            "-c",
            "-o", dest.name,
            "-d", str(dest.parent),
            url,
        ])
    else:
        run_cmd(["wget", "-q", "--show-progress", "-O", dest, url])


def install_mmseqs2_static():
    if shutil.which("mmseqs"):
        print("mmseqs already available:", shutil.which("mmseqs"))
        return

    if platform.machine() not in {"x86_64", "AMD64"}:
        raise RuntimeError("This Colab install cell currently expects x86_64 Linux.")

    if has_cpu_flag("avx2"):
        archive_name = "mmseqs-linux-avx2.tar.gz"
    elif has_cpu_flag("sse4_1"):
        archive_name = "mmseqs-linux-sse41.tar.gz"
    else:
        archive_name = "mmseqs-linux-sse2.tar.gz"

    archive = WORK_ROOT / archive_name
    download(f"https://mmseqs.com/latest/{archive_name}", archive)

    extract_dir = WORK_ROOT / "mmseqs"
    if not extract_dir.exists():
        run_cmd(["tar", "xzf", archive, "-C", WORK_ROOT])

    src = extract_dir / "bin" / "mmseqs"
    dst = TOOLS_DIR / "mmseqs"
    if not dst.exists():
        dst.symlink_to(src)

    print("mmseqs installed:", shutil.which("mmseqs"))


def install_foldseek_static():
    if shutil.which("foldseek"):
        print("foldseek already available:", shutil.which("foldseek"))
        return

    if platform.machine() not in {"x86_64", "AMD64"}:
        raise RuntimeError("This Colab install cell currently expects x86_64 Linux.")

    if not has_cpu_flag("avx2"):
        raise RuntimeError(
            "Foldseek's recommended Colab-style static binary is AVX2. "
            "This CPU does not report AVX2; use conda fallback for foldseek."
        )

    archive_name = "foldseek-linux-avx2.tar.gz"
    archive = WORK_ROOT / archive_name
    download(f"https://mmseqs.com/foldseek/{archive_name}", archive)

    extract_dir = WORK_ROOT / "foldseek"
    if not extract_dir.exists():
        run_cmd(["tar", "xzf", archive, "-C", WORK_ROOT])

    src = extract_dir / "bin" / "foldseek"
    dst = TOOLS_DIR / "foldseek"
    if not dst.exists():
        dst.symlink_to(src)

    print("foldseek installed:", shutil.which("foldseek"))


def install_python_package():
    # Direct install from GitHub branch/ref. Faster and cleaner than clone + checkout + pip install .
    pkg_url = f"git+{REPO_URL}@{REPO_REF}"
    run_cmd([
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        "--no-cache-dir",
        pkg_url,
    ])


def get_package_root(package_name="meetneighbors") -> Path:
    spec = importlib.util.find_spec(package_name)
    if spec is None or spec.origin is None:
        raise RuntimeError(f"Could not find installed package: {package_name}")
    return Path(spec.origin).resolve().parent


def install_glm_model():
    package_root = get_package_root("meetneighbors")
    model_dir = package_root / "predictvfs" / "glm" / "model"
    model_dir.mkdir(parents=True, exist_ok=True)

    model_file = model_dir / "glm.bin"
    if model_file.exists() and model_file.stat().st_size > 0:
        print("gLM model already present:", model_file)
        return

    tmp_model = WORK_ROOT / "glm.bin"
    download("https://zenodo.org/records/7855545/files/glm.bin?download=1", tmp_model)

    shutil.copy2(tmp_model, model_file)
    print("gLM model installed:", model_file)


install_mmseqs2_static()
install_foldseek_static()
install_python_package()
install_glm_model()

os.environ["MPLBACKEND"] = "Agg" #matplotlib error fix


missing_tools = [
    tool for tool in ("meetneighbors", "mmseqs", "foldseek")
    if shutil.which(tool) is None
]
if missing_tools:
    raise RuntimeError(
        "Install finished, but these tools were not found on PATH: "
        + ", ".join(missing_tools)
    )

print("Install complete.")
print("meetneighbors:", shutil.which("meetneighbors"))
print("mmseqs:", shutil.which("mmseqs"))
print("foldseek:", shutil.which("foldseek"))

run_cmd(["mmseqs", "version"])
run_cmd(["foldseek", "version"])

mmseqs already available: /usr/local/bin/mmseqs
foldseek already available: /usr/local/bin/foldseek
$ /usr/bin/python3 -m pip install --upgrade --no-cache-dir git+https://github.com/mcn3159/meet-the-neighbors.git@colab_run
gLM model already present: /usr/local/lib/python3.12/dist-packages/meetneighbors/predictvfs/glm/model/glm.bin
Install complete.
meetneighbors: /usr/local/bin/meetneighbors
mmseqs: /usr/local/bin/mmseqs
foldseek: /usr/local/bin/foldseek
$ mmseqs version
$ foldseek version


In [4]:
!meetneighbors predictvf --help

usage: neighbors predictvf [-h] [--query_fasta QUERY_FASTA] [--seq_id SEQ_ID]
                           [--cov COV] [--mem MEM] [--threads THREADS]
                           [--genomes GENOMES] [--genome_tsv GENOME_TSV]
                           --out OUT [--genomes_db GENOMES_DB]
                           [--neighborhood_size NEIGHBORHOOD_SIZE]
                           [--min_prots MIN_PROTS] [--max_prots MAX_PROTS]
                           [-ig INTERGENIC] [--red_olp]
                           [--olp_window OLP_WINDOW] [-ho] [--remove_temp]
                           [--resume] [--gpu GPU] [--cluster]
                           [--foldseek_structs FOLDSEEK_STRUCTS]
                           [--tmcutoff TMCUTOFF]
                           [--fs_qcovcutoff FS_QCOVCUTOFF]
                           [--lddtcutoff LDDTCUTOFF] [--include_structhits]
                           [--prot_genome_pairs PROT_GENOME_PAIRS]
                           [--glm_bs GLM_BS] [--memory_optimize]

## Upload genome files

Drag and drop all files together when prompted.

Each genome must have exactly one `.gff` and exactly one matching protein FASTA
file. Protein FASTA files may end in `.faa` or `.fasta`.

The notebook pairs files by basename, so `GenomeA.gff` pairs with
`GenomeA.faa`, and `GenomeB.gff` pairs with `GenomeB.fasta`.

In [5]:
#@title 2. Upload `.gff` and `.faa`/`.fasta` genome pairs
#@markdown Click the upload button and select all genome files for this run.
#@markdown Upload no more than `3` `.gff` files and their matching
#@markdown protein FASTA files.

try:
    from google.colab import files
except ImportError as exc:
    raise RuntimeError(
        "This upload cell is intended for Google Colab, where "
        "`google.colab.files.upload()` is available."
    ) from exc

if UPLOAD_DIR.exists():
    shutil.rmtree(UPLOAD_DIR)
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()
if not uploaded:
    raise ValueError("No files were uploaded.")

uploaded_paths = []
for uploaded_name in uploaded:
    source = Path(uploaded_name)
    normalized_name = source.stem + source.suffix.lower()
    destination = UPLOAD_DIR / normalized_name
    if destination.exists():
        raise ValueError(
            f"Duplicate uploaded filename after extension normalization: "
            f"{destination.name}"
        )
    shutil.move(str(source), destination)
    uploaded_paths.append(destination)

print("Uploaded files:")
for path in uploaded_paths:
    print(f"  {path.name}")

#Validate genome pairs and build `genome_pairs.tsv`
#this creates the three-column TSV used by `meetneighbors --genome_tsv`.
#The columns are genome name, protein FASTA path, and GFF path.

PROTEIN_EXTENSIONS = {".faa", ".fasta"}
SUPPORTED_EXTENSIONS = PROTEIN_EXTENSIONS | {".gff"}


def uploaded_file_stem(path):
    suffix = path.suffix.lower()
    return path.name[: -len(suffix)]

# print(f"Detected {len(genome_pairs)} genome pair(s):")
# for genome_name, protein_path, gff_path in genome_pairs:
#     print(f"  {genome_name}: {gff_path.name} + {protein_path.name}")


Saving NR2335.faa to NR2335.faa
Saving NR2335.gff to NR2335.gff
Uploaded files:
  NR2335.faa
  NR2335.gff


## Run Meet-the-Neighbors

This step can take a while because it computes protein language model and
genomic language model embeddings. GPU runtimes are recommended when available.

The command uses `predictvf` in genome-only mode through `--genome_tsv`.

In [11]:
#@title 2. Run Meet-the-Neighbors

import os
import shutil
import subprocess
from pathlib import Path

try:
    import torch
    gpu = 1 if torch.cuda.is_available() else 0
except Exception:
    gpu = 0

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

predict_command = [
    "meetneighbors",
    "predictvf",
    "--genomes", str(UPLOAD_DIR),
    "--out", str(OUTPUT_DIR),
    "--threads", str(10),
    "--mem", str(mem_gb),
    "--glm_bs", str(glm_batch_size),
    "--gpu", str(gpu),
    "--memory_optimize",
    "--remove_temp",
    "--cluster",
    "-ns", "30000",
    "-ig", "10000",
]

# Only resume if the output directory already has content, not merely because it exists.
if OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir()):
    predict_command.append("--resume")

print("Running Meet-the-Neighbors with this command:")
print(" ".join(str(part) for part in predict_command))

process = subprocess.Popen(
    predict_command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=os.environ.copy(),
)

for line in process.stdout:
    print(line, end="")

return_code = process.wait()
if return_code != 0:
    raise subprocess.CalledProcessError(return_code, predict_command)

Running Meet-the-Neighbors with this command:
meetneighbors predictvf --genomes /content/meetneighbors_colab/uploaded_genomes --out /content/meetneighbors_colab/run --threads 10 --mem 12 --glm_bs 50 --gpu 1 --memory_optimize --remove_temp --cluster -ns 30000 -ig 10000 --resume
/usr/local/lib/python3.13/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator LabelBinarizer from version 1.1.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator PCA from version 1.1.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#s

CalledProcessError: Command '['meetneighbors', 'predictvf', '--genomes', '/content/meetneighbors_colab/uploaded_genomes', '--out', '/content/meetneighbors_colab/run', '--threads', '10', '--mem', '12', '--glm_bs', '50', '--gpu', '1', '--memory_optimize', '--remove_temp', '--cluster', '-ns', '30000', '-ig', '10000', '--resume']' returned non-zero exit status 1.

In [10]:
!ls /usr/local/lib/python3.13/site-packages/meetneighbors/predictvfs/glm/model/

glm.bin  __init__.py  __pycache__


## Download the final prediction table

This notebook intentionally keeps only one output file:

`neighborhood_based_predictions.tsv`

All other pipeline files are treated as intermediates.

In [ ]:
#@title 6. Save, preview, and download `neighborhood_based_predictions.tsv`
#@markdown The final TSV is copied into a small results directory before
#@markdown cleanup starts.

import pandas as pd

pipeline_output = OUTPUT_DIR / "neighborhood_based_predictions.tsv"
if not pipeline_output.exists():
    raise FileNotFoundError(
        f"Expected output was not found: {pipeline_output}. "
        "Check the previous cell logs for the pipeline error."
    )

FINAL_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2(pipeline_output, FINAL_OUTPUT)

print(f"Final output saved to {FINAL_OUTPUT}")
preview_df = pd.read_csv(FINAL_OUTPUT, sep="\t")
print(f"Rows: {preview_df.shape[0]}, columns: {preview_df.shape[1]}")
display(preview_df.head())

files.download(str(FINAL_OUTPUT))

## Cleanup

This cell removes uploaded genome files, the cloned repository, and pipeline
intermediate files. The final TSV remains available at:

`/content/meetneighbors_colab/final_results/neighborhood_based_predictions.tsv`

Rerun the install and upload cells if you want to start a new analysis.

In [ ]:
#@title 7. Clean up intermediate files
#@markdown After this cell, only the copied final TSV is retained under
#@markdown `/content/meetneighbors_colab/final_results`.

for path in (UPLOAD_DIR, RUN_DIR, REPO_DIR):
    if path.exists():
        shutil.rmtree(path)

print("Cleanup complete.")
print(f"Kept final output: {FINAL_OUTPUT}")